# Trajectory 품질 분석

학습 데이터 품질 검증을 위한 분석:
- **MC=0.0 케이스**: Final MC가 0.0인 케이스 상세 분석
- **Category A**: 가짜 정답 의심군 (False Positives)
- **Category B**: 학습 데이터로 쓰면 안 되는 행동 (Bad Trajectory)
- **Category C**: Step 1 억까 피해 규모

In [1]:
import json
from IPython.display import display, HTML
import re

In [2]:
# Load all trajectories
trajectories = []
with open('../outputs/test_2q_dynamic/results_hybrid_20251211_161159.jsonl', 'r') as f:
    for line in f:
        trajectories.append(json.loads(line))

print(f"✓ Loaded {len(trajectories)} trajectories")

✓ Loaded 89 trajectories


In [ ]:
# Helper functions

def truncate(text, max_len=150):
    """Truncate text with ellipsis."""
    if len(text) > max_len:
        return text[:max_len] + "..."
    return text

def get_step_type(step):
    """Get step type from action field."""
    action = step.get('action', 'Unknown')
    if action == 'Search':
        return 'rag'
    elif action == 'Reason':
        return 'cot'
    elif action == 'Finish':
        return 'finish'
    else:
        return 'unknown'

def display_trajectory(traj, index, total, category_name=""):
    """Display a single trajectory in nice format."""
    
    # Header
    html = f"""
    <div style="border: 2px solid #333; padding: 20px; margin: 20px 0; border-radius: 10px; background-color: #f9f9f9;">
        <h2 style="color: #d00;">📋 {category_name} Case #{index + 1} / {total}</h2>
        <p><strong>Question ID:</strong> {traj['question_id']}</p>
        <p><strong>Correct:</strong> {'✅ YES' if traj['is_correct'] else '❌ NO'}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #fff;">
        <h3>🔍 Question</h3>
        <p style="font-size: 16px; line-height: 1.6;">{traj['question']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #e8f5e9;">
        <h3>✅ Gold Answer</h3>
        <p style="font-size: 16px; font-weight: bold;">{traj['gold_answer']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: {'#ffebee' if not traj['is_correct'] else '#e8f5e9'};">
        <h3>🤖 Predicted Answer</h3>
        <p style="font-size: 16px;">{traj['predicted_answer']}</p>
    </div>
    
    <div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #fff3e0;">
        <h3>📊 Summary</h3>
        <p><strong>Total Steps:</strong> {traj['num_steps']} (CoT: {traj['num_cot_steps']}, RAG: {traj['num_rag_steps']})</p>
        <p><strong>Has RAG:</strong> {'Yes' if traj['has_rag'] else 'No'}</p>
        <p><strong>Final MC:</strong> <span style="color: {'red' if traj['steps'][-1]['mc_after'] == 0.0 else 'green'}; font-weight: bold;">{traj['steps'][-1]['mc_after']:.3f}</span></p>
    </div>
    """
    
    # Steps
    html += '<div style="border: 1px solid #666; padding: 15px; margin: 10px 0; background-color: #f5f5f5;">'
    html += '<h3>🔄 Step-by-Step Trajectory</h3>'
    
    for step in traj['steps']:
        step_type = get_step_type(step)
        step_color = '#e3f2fd' if step_type == 'rag' else '#fff9c4' if step_type == 'cot' else '#c8e6c9' if step_type == 'finish' else '#f0f0f0'
        label_color = 'green' if step['label'] == 'good' else 'red'
        
        html += f'<div style="border-left: 4px solid {"#2196F3" if step_type == "rag" else "#FFC107" if step_type == "cot" else "#4CAF50"}; padding: 10px; margin: 10px 0; background-color: {step_color};">'
        html += f'<h4>Step {step["step_num"]} - {step.get("action", "Unknown").upper()}</h4>'
        html += f'<p><strong>MC:</strong> {step["mc_before"]:.3f} → {step["mc_after"]:.3f} '
        html += f'(Δ {step["mc_after"] - step["mc_before"]:+.3f})</p>'
        html += f'<p><strong>RPE:</strong> <span style="color: {"red" if step["rpe"] > 10 else "black"}">{step["rpe"]:.3f}</span> | '
        html += f'<strong>Label:</strong> <span style="color: {label_color}; font-weight: bold;">{step["label"].upper()}</span></p>'
        
        if step_type == 'rag':
            query = step.get('action_input')
            if query:
                html += f'<p><strong>🔍 Query:</strong> <code>{query}</code></p>'
            
            # Show passage titles
            if step.get('passage_titles'):
                html += f'<p><strong>📚 Passages:</strong> {", ".join(step["passage_titles"][:3])}...</p>'
        
        # Show content
        html += f'<details><summary><strong>💬 Content (click to expand)</strong></summary>'
        html += f'<pre style="white-space: pre-wrap; background-color: #fff; padding: 10px; border-radius: 5px;">{step["content"]}</pre>'
        html += f'</details>'
        
        html += '</div>'
    
    html += '</div>'
    
    display(HTML(html))

---
## MC=0.0 케이스 분석

Final MC가 0.0인 케이스들을 상세 분석합니다.

In [ ]:
# MC=0.0 케이스 필터링
mc_zero_cases = []

for traj in trajectories:
    if traj['steps']:
        final_mc = traj['steps'][-1]['mc_after']
        if final_mc == 0.0:
            mc_zero_cases.append(traj)

print(f"✓ Found {len(mc_zero_cases)} trajectories with MC=0.0")
print(f"  Percentage: {len(mc_zero_cases)/len(trajectories)*100:.1f}%")
print()

# MC=0.0 케이스들의 정답률
if mc_zero_cases:
    correct_count = sum(1 for traj in mc_zero_cases if traj['is_correct'])
    incorrect_count = len(mc_zero_cases) - correct_count
    
    print("MC=0.0 케이스 통계")
    print("=" * 50)
    print(f"✅ 정답: {correct_count} ({correct_count/len(mc_zero_cases)*100:.1f}%)")
    print(f"❌ 오답: {incorrect_count} ({incorrect_count/len(mc_zero_cases)*100:.1f}%)")
    print()
    
    # MC 변화 패턴
    always_zero = sum(1 for traj in mc_zero_cases if all(step['mc_after'] == 0.0 for step in traj['steps']))
    became_zero = len(mc_zero_cases) - always_zero
    
    print("MC 변화 패턴")
    print(f"  항상 0.0: {always_zero}")
    print(f"  중간에 0.0이 됨: {became_zero}")
else:
    print("MC=0.0 케이스가 없습니다.")

In [ ]:
# MC=0.0 케이스 목록
print(f"MC=0.0 케이스 목록 ({len(mc_zero_cases)}개)")
print("=" * 50)

for i, traj in enumerate(mc_zero_cases[:10]):
    print(f"{i+1}. {traj['question_id']} | Correct: {'✅' if traj['is_correct'] else '❌'} | Steps: {traj['num_steps']}")
    print(f"   Q: {truncate(traj['question'], 70)}")
    print(f"   Gold: {traj['gold_answer']}")
    print(f"   Pred: {truncate(str(traj['predicted_answer']), 50)}")
    print()

In [ ]:
# MC=0.0 케이스 상세 보기
mc_zero_index = 0

if mc_zero_cases and mc_zero_index < len(mc_zero_cases):
    display_trajectory(mc_zero_cases[mc_zero_index], mc_zero_index, len(mc_zero_cases), "MC=0.0")
else:
    print("MC=0.0 케이스가 없습니다.")

---
## Category A: 가짜 정답 의심군 (False Positives)

**조건**: `is_correct=True` 이지만 `final_mc=0.0`

MC가 0인데 정답이라고? 우연히 맞춘 거일 가능성이 높습니다.

In [4]:
# Category A: 정답인데 MC=0.0
category_a = []

for traj in trajectories:
    if traj['steps']:
        final_mc = traj['steps'][-1]['mc_after']
        if traj['is_correct'] and final_mc == 0.0:
            category_a.append(traj)

print(f"Category A: 가짜 정답 의심군")
print(f"="*50)
print(f"총 {len(category_a)}개 ({len(category_a)/len(trajectories)*100:.1f}%)")
print()

for i, traj in enumerate(category_a[:5]):
    print(f"{i+1}. {traj['question_id']}")
    print(f"   Q: {truncate(traj['question'], 60)}")
    print(f"   Gold: {traj['gold_answer']}")
    print(f"   Pred: {truncate(traj['predicted_answer'], 60)}")
    print()

Category A: 가짜 정답 의심군
총 3개 (3.4%)

1. 5ae3b4d05542992f92d82349
   Q: Who is the director of the 2003 film which has scenes in it ...
   Gold: Todd Phillips
   Pred: Todd Phillips

2. 5a760a5f554299109176e629
   Q: What American country music singer-songwriter, born in May o...
   Gold: Tammy Wynette
   Pred: Tammy Wynette

3. 5ac1b48f55429963665198f3
   Q: The Golden Globe Award winner for best actor from "Roseanne"...
   Gold: Zooey Deschanel
   Pred: Zooey Deschanel



In [5]:
# Category A 상세 보기
cat_a_index = 0

if category_a and cat_a_index < len(category_a):
    display_trajectory(category_a[cat_a_index], cat_a_index, len(category_a), "Category A")
else:
    print("Category A 케이스가 없습니다.")

---
## Category B: 학습 데이터로 쓰면 안 되는 행동 (Bad Trajectory)

정답 여부와 상관없이, 과정이 엉망인 케이스들. 학습 데이터에서 **제외(Filtering)**해야 합니다.

- **조건 1**: 무한 루프 (Step 수가 10개)
- **조건 2**: RPE 이상치 (RPE > 10.0)

In [6]:
# Category B: Bad Trajectory
category_b_loop = []  # 무한 루프
category_b_rpe = []   # RPE 이상치

for traj in trajectories:
    # 조건 1: 무한 루프 (Step 수가 10개)
    if traj['num_steps'] >= 10:
        category_b_loop.append(traj)
    
    # 조건 2: RPE 이상치
    for step in traj['steps']:
        if step['rpe'] > 10.0:
            category_b_rpe.append(traj)
            break  # 하나만 있어도 추가

# 중복 제거한 전체 Category B
category_b_ids = set(t['question_id'] for t in category_b_loop) | set(t['question_id'] for t in category_b_rpe)
category_b = [t for t in trajectories if t['question_id'] in category_b_ids]

print(f"Category B: 학습 데이터로 쓰면 안 되는 행동")
print(f"="*50)
print(f"B-1. 무한 루프 (steps >= 10): {len(category_b_loop)}개")
print(f"B-2. RPE 이상치 (RPE > 10): {len(category_b_rpe)}개")
print(f"총 (중복 제거): {len(category_b)}개 ({len(category_b)/len(trajectories)*100:.1f}%)")
print()

# 정답률
correct_in_b = sum(1 for t in category_b if t['is_correct'])
print(f"Category B 내 정답률: {correct_in_b}/{len(category_b)} ({correct_in_b/len(category_b)*100:.1f}% if len(category_b) > 0 else 0)")

Category B: 학습 데이터로 쓰면 안 되는 행동
B-1. 무한 루프 (steps >= 10): 32개
B-2. RPE 이상치 (RPE > 10): 19개
총 (중복 제거): 43개 (48.3%)

Category B 내 정답률: 14/43 (32.6% if len(category_b) > 0 else 0)


In [7]:
# B-1: 무한 루프 케이스 목록
print(f"B-1. 무한 루프 케이스 ({len(category_b_loop)}개)")
print(f"="*50)

for i, traj in enumerate(category_b_loop[:10]):
    print(f"{i+1}. {traj['question_id']} | Steps: {traj['num_steps']} | Correct: {'✅' if traj['is_correct'] else '❌'}")
    print(f"   Q: {truncate(traj['question'], 70)}")
    print()

B-1. 무한 루프 케이스 (32개)
1. 5a82171f5542990a1d231f4a | Steps: 10 | Correct: ❌
   Q:  What nationality was James Henry Miller's wife?

2. 5a84dd955542997b5ce3ff79 | Steps: 10 | Correct: ❌
   Q: Cadmium Chloride is slightly soluble in this chemical, it is also call...

3. 5a7e36045542991319bc9440 | Steps: 10 | Correct: ✅
   Q: Which tennis player won more Grand Slam titles, Henri Leconte or Jonat...

4. 5aba66c855429939ce03dcdb | Steps: 10 | Correct: ❌
   Q: Gunmen from Laredo starred which narrator of "Frontier"?

5. 5a7722d655429966f1a36c99 | Steps: 10 | Correct: ✅
   Q: Where did the form of music played by Die Rhöner Säuwäntzt originate?

6. 5ab381b155429969a97a816b | Steps: 10 | Correct: ❌
   Q: What U.S Highway gives access to Zilpo Road, and is also known as Midl...

7. 5ae0d91e55429924de1b7198 | Steps: 10 | Correct: ❌
   Q: The 1988 American comedy film, The Great Outdoors, starred a four-time...

8. 5ac2a5d455429921a00ab01b | Steps: 10 | Correct: ❌
   Q: What are the names of the cu

In [8]:
# B-2: RPE 이상치 케이스 목록
print(f"B-2. RPE 이상치 케이스 ({len(category_b_rpe)}개)")
print(f"="*50)

for i, traj in enumerate(category_b_rpe[:10]):
    # 가장 높은 RPE 찾기
    max_rpe_step = max(traj['steps'], key=lambda s: s['rpe'])
    print(f"{i+1}. {traj['question_id']} | Max RPE: {max_rpe_step['rpe']:.2f} (Step {max_rpe_step['step_num']}) | Correct: {'✅' if traj['is_correct'] else '❌'}")
    print(f"   Q: {truncate(traj['question'], 70)}")
    print()

B-2. RPE 이상치 케이스 (19개)
1. 5a8d7341554299441c6b9fe5 | Max RPE: 15.62 (Step 2) | Correct: ❌
   Q: Musician and satirist Allie Goertz wrote a song about the "The Simpson...

2. 5a7722d655429966f1a36c99 | Max RPE: 31.25 (Step 1) | Correct: ✅
   Q: Where did the form of music played by Die Rhöner Säuwäntzt originate?

3. 5adf732a5542993a75d264e9 | Max RPE: 84.38 (Step 1) | Correct: ✅
   Q: Which  American politician did Donahue replaced 

4. 5ae3b4d05542992f92d82349 | Max RPE: 100.00 (Step 2) | Correct: ✅
   Q: Who is the director of the 2003 film which has scenes in it filmed at ...

5. 5a7d90195542991319bc93cf | Max RPE: 12.50 (Step 1) | Correct: ❌
   Q: The axial turbojet Pirna 014 was designed by engineers from this Germa...

6. 5a760a5f554299109176e629 | Max RPE: 87.50 (Step 2) | Correct: ✅
   Q: What American country music singer-songwriter, born in May of 1942, sa...

7. 5a8195f35542995ce29dcc0c | Max RPE: 21.88 (Step 2) | Correct: ❌
   Q: A Head Full of Dreams Tour is the seventh to

In [9]:
# Category B 상세 보기 - 무한 루프
cat_b_loop_index = 0

if category_b_loop and cat_b_loop_index < len(category_b_loop):
    display_trajectory(category_b_loop[cat_b_loop_index], cat_b_loop_index, len(category_b_loop), "Category B-1 (무한루프)")
else:
    print("Category B-1 케이스가 없습니다.")

In [10]:
# Category B 상세 보기 - RPE 이상치
cat_b_rpe_index = 0

if category_b_rpe and cat_b_rpe_index < len(category_b_rpe):
    display_trajectory(category_b_rpe[cat_b_rpe_index], cat_b_rpe_index, len(category_b_rpe), "Category B-2 (RPE이상치)")
else:
    print("Category B-2 케이스가 없습니다.")

---
## Category C: Step 1 억까 피해 규모

"Step 1에서 생각 좀 했다고 Bad 주는 경우"가 얼마나 되는지 확인합니다.

**조건**: Step 1의 `label='bad'`이고, `action='Reason'`

In [ ]:
# Category C: Step 1 억까
category_c = []

for traj in trajectories:
    if traj['steps']:
        step1 = traj['steps'][0]
        # Step 1이 Reason이고 Bad인 경우
        if step1.get('action') == 'Reason' and step1.get('label') == 'bad':
            category_c.append(traj)

print(f"Category C: Step 1 억까 피해 규모")
print(f"="*50)
print(f"총 {len(category_c)}개 ({len(category_c)/len(trajectories)*100:.1f}%)")
print()

if category_c:
    # 이 중 최종 정답률
    correct_in_c = sum(1 for t in category_c if t['is_correct'])
    print(f"Step 1 억까 후 최종 정답률: {correct_in_c}/{len(category_c)} ({correct_in_c/len(category_c)*100:.1f}%)")
    print()
    
    # Step 1 RPE 분포
    step1_rpes = [t['steps'][0]['rpe'] for t in category_c]
    print(f"Step 1 RPE 분포:")
    print(f"  Min: {min(step1_rpes):.3f}")
    print(f"  Max: {max(step1_rpes):.3f}")
    print(f"  Avg: {sum(step1_rpes)/len(step1_rpes):.3f}")
else:
    print("Step 1 억까 케이스가 없습니다.")

In [12]:
# Category C 목록
print(f"Category C: Step 1 억까 케이스 ({len(category_c)}개)")
print(f"="*50)

for i, traj in enumerate(category_c[:10]):
    step1 = traj['steps'][0]
    print(f"{i+1}. {traj['question_id']}")
    print(f"   Step1 RPE: {step1['rpe']:.3f} | MC: {step1['mc_before']:.3f} → {step1['mc_after']:.3f}")
    print(f"   Final: {'✅' if traj['is_correct'] else '❌'} | Steps: {traj['num_steps']}")
    print(f"   Q: {truncate(traj['question'], 60)}")
    print()

Category C: Step 1 억까 케이스 (0개)


In [13]:
# Category C 상세 보기
cat_c_index = 0

if category_c and cat_c_index < len(category_c):
    display_trajectory(category_c[cat_c_index], cat_c_index, len(category_c), "Category C (Step1 억까)")
else:
    print("Category C 케이스가 없습니다.")

Category C 케이스가 없습니다.


---
## 전체 통계 요약

In [14]:
print("="*60)
print("전체 통계 요약")
print("="*60)
print(f"총 Trajectory: {len(trajectories)}개")
print(f"전체 정답률: {sum(1 for t in trajectories if t['is_correct'])}/{len(trajectories)} ({sum(1 for t in trajectories if t['is_correct'])/len(trajectories)*100:.1f}%)")
print()
print(f"Category A (가짜 정답 의심): {len(category_a)}개 ({len(category_a)/len(trajectories)*100:.1f}%)")
print(f"Category B (Bad Trajectory): {len(category_b)}개 ({len(category_b)/len(trajectories)*100:.1f}%)")
print(f"  - B-1 무한루프: {len(category_b_loop)}개")
print(f"  - B-2 RPE이상치: {len(category_b_rpe)}개")
print(f"Category C (Step1 억까): {len(category_c)}개 ({len(category_c)/len(trajectories)*100:.1f}%)")
print()

# 필터링 후 남는 데이터
bad_ids = set(t['question_id'] for t in category_a) | set(t['question_id'] for t in category_b)
clean_data = [t for t in trajectories if t['question_id'] not in bad_ids]
print(f"필터링 후 (A, B 제외): {len(clean_data)}개 ({len(clean_data)/len(trajectories)*100:.1f}%)")
clean_correct = sum(1 for t in clean_data if t['is_correct'])
print(f"필터링 후 정답률: {clean_correct}/{len(clean_data)} ({clean_correct/len(clean_data)*100:.1f}% if len(clean_data) > 0 else 0)")

전체 통계 요약
총 Trajectory: 89개
전체 정답률: 46/89 (51.7%)

Category A (가짜 정답 의심): 3개 (3.4%)
Category B (Bad Trajectory): 43개 (48.3%)
  - B-1 무한루프: 32개
  - B-2 RPE이상치: 19개
Category C (Step1 억까): 0개 (0.0%)

필터링 후 (A, B 제외): 46개 (51.7%)
필터링 후 정답률: 32/46 (69.6% if len(clean_data) > 0 else 0)


---
## 네비게이션 헬퍼

In [ ]:
# 전체 trajectory 중 특정 인덱스 보기
all_index = 0

if all_index < len(trajectories):
    display_trajectory(trajectories[all_index], all_index, len(trajectories), "All")
else:
    print(f"Index out of range (0 ~ {len(trajectories)-1})")

In [ ]:
# Question ID로 검색
search_id = "5a8d7341554299441c6b9fe5"  # 원하는 ID 입력

found = [t for t in trajectories if t['question_id'] == search_id]
if found:
    display_trajectory(found[0], 0, 1, "Search Result")
else:
    print(f"Question ID '{search_id}' not found")